# 04 Simple Training

Быстрое обучение без Optuna. Параметры взяты из прошлых удачных запусков 01_test_training.

In [1]:
from pathlib import Path
import random
import sys

import joblib
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.models.training import train_direction_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

price_df, files = load_all_price_data(PROJECT_ROOT / "data")
print(f"loaded files={len(files)}, rows={len(price_df)}")

loaded files=45, rows=277105


## Настройки

Меняй только верхний блок. По умолчанию обучаем event GRU и event LSTM.

In [2]:
RUN_GRU_EVENT = True
RUN_LSTM_EVENT = True
RUN_GRU_FULL = False
RUN_LSTM_FULL = False

SELECTION_METRIC = "balanced_accuracy"  # accuracy / balanced_accuracy / f1
EPOCHS = 40
LABEL_THRESHOLD = config.DEFAULT_LABEL_THRESHOLD
HORIZON = config.DEFAULT_HORIZON_CANDLES

PARAMS = {
    "gru": {
        "learning_rate": 0.0028,
        "hidden_size": 128,
        "dropout": 0.20,
        "num_layers": 3,
        "batch_size": 128,
    },
    "lstm": {
        "learning_rate": 0.0020,
        "hidden_size": 128,
        "dropout": 0.45,
        "num_layers": 2,
        "batch_size": 64,
    },
}

In [3]:
def get_paths(model_type: str, event_only: bool):
    if event_only:
        return (
            config.EVENT_MODEL_PATHS[model_type],
            config.EVENT_SCALER_PATHS[model_type],
            config.EVENT_CONFIG_PATHS[model_type],
        )
    return (
        config.FULL_MODEL_PATHS[model_type],
        config.FULL_SCALER_PATHS[model_type],
        config.FULL_CONFIG_PATHS[model_type],
    )


def train_one(model_type: str, event_only: bool):
    params = PARAMS[model_type]
    model_path, scaler_path, config_path = get_paths(model_type, event_only)

    result = train_direction_model(
        price_df=price_df,
        model_type=model_type,
        event_only=event_only,
        label_threshold=LABEL_THRESHOLD,
        horizon=HORIZON,
        epochs=EPOCHS,
        selection_metric=SELECTION_METRIC,
        model_path=model_path,
        scaler_path=scaler_path,
        **params,
    )

    model_config = {
        "model_type": model_type,
        "input_size": len(config.FEATURE_COLUMNS),
        "hidden_size": params["hidden_size"],
        "dropout": params["dropout"],
        "num_layers": params["num_layers"],
        "selection_metric": SELECTION_METRIC,
        "label_threshold": LABEL_THRESHOLD,
        "horizon": HORIZON,
        "feature_columns": config.FEATURE_COLUMNS,
    }
    config_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model_config, config_path)

    row = {
        "model": model_type,
        "mode": "event" if event_only else "full",
        "best_valid_metric": SELECTION_METRIC,
        "best_valid_score": result["best_valid_score"],
        "test_accuracy": result["test_metrics"]["accuracy"],
        "test_balanced_accuracy": result["test_metrics"]["balanced_accuracy"],
        "test_f1": result["test_metrics"]["f1"],
        "train_samples": result["train_samples"],
        "valid_samples": result["valid_samples"],
        "test_samples": result["test_samples"],
        "model_path": str(model_path),
        "scaler_path": str(scaler_path),
        "config_path": str(config_path),
    }
    return row

In [4]:
jobs = []
if RUN_GRU_EVENT:
    jobs.append(("gru", True))
if RUN_LSTM_EVENT:
    jobs.append(("lstm", True))
if RUN_GRU_FULL:
    jobs.append(("gru", False))
if RUN_LSTM_FULL:
    jobs.append(("lstm", False))

rows = []
for model_type, event_only in jobs:
    print(f"Training {model_type.upper()} / {'event' if event_only else 'full'}")
    rows.append(train_one(model_type, event_only))

results_df = pd.DataFrame(rows)
display(results_df)

Training GRU / event
Training LSTM / event


,model,mode,best_valid_metric,best_valid_score,test_accuracy,test_balanced_accuracy,test_f1,train_samples,valid_samples,test_samples,model_path,scaler_path,config_path
0,gru,event,balanced_accuracy,0.527649,0.524070,0.524163,0.527512,10172,5220,3843,/workspace/data/models/gru_event_direction_bes...,/workspace/data/models/gru_event_direction_sca...,/workspace/data/models/gru_event_direction_con...
1,lstm,event,balanced_accuracy,0.531249,0.521468,0.520700,0.468651,10172,5220,3843,/workspace/data/models/lstm_event_direction_be...,/workspace/data/models/lstm_event_direction_sc...,/workspace/data/models/lstm_event_direction_co...
